<a href="https://colab.research.google.com/github/Stdcoders/Graph-RAG/blob/main/GraphRAG_L7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/genaiconference/Agentic_KAG_Workshop_DHS_2026.git

Cloning into 'Agentic_KAG_Workshop_DHS_2026'...
remote: Enumerating objects: 403, done.
remote: Counting objects: 100% (179/179), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 403 (delta 125), reused 57 (delta 55), pack-reused 224 (from 2)
Receiving objects: 100% (403/403), 13.34 MiB | 11.45 MiB/s, done.
Resolving deltas: 100% (217/217), done.


In [ ]:
import os

os.chdir('/content/Agentic_KAG_Workshop_DHS_2026/')

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    print("error reading env details")
    pass

# --- Neo4j Sandbox ---
NEO4J_URI      = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE')

# --- OpenAI ---
os.environ.setdefault(
    'NVIDIA_API_KEY',
    os.getenv('NVIDIA_API_KEY')
)
OPENAI_API_KEY = os.getenv('NVIDIA_API_KEY') # Define OPENAI_API_KEY as a Python variable

# Tavily web-search API key (used by the Web Search tool for latest/recent movies)
TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')

print('NEO4J_URI :', NEO4J_URI)
print('OPENAI key set:', bool(os.environ.get('NVIDIA_API_KEY')))

NEO4J_URI : neo4j+s://313964e6.databases.neo4j.io
OPENAI key set: True


In [ ]:
!pip install -r /content/Agentic_KAG_Workshop_DHS_2026/requirements.txt --quiet

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.7/263.7 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.2/58.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.0/358.0 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.8/85.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 669.4/669.4 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.0 MB/s eta 0:00:00
   ━━━━━

In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)

# (Optional) Test the connection
driver.verify_connectivity()

In [ ]:
%pip install -U langchain-nvidia-ai-endpoints

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 5.8 MB/s eta 0:00:00


In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from neo4j_graphrag.embeddings.openai import BaseOpenAIEmbeddings
NVIDIA_BASE_URL = "https://integrate.api.nvidia.com/v1"

class NemotronEmbeddings(BaseOpenAIEmbeddings):
    """
    NVIDIA Nemotron embeddings via NIM's OpenAI-compatible /v1/embeddings endpoint.
    Nemotron-family embedding models require an `input_type` of "passage" (indexing)
    or "query" (retrieval) on every request, so we inject it automatically here.
    """
    def __init__(self, model="nvidia/nemotron-3-embed-1b", input_type="passage", **kwargs):
        self.input_type = input_type
        super().__init__(model=model, **kwargs)

    def _initialize_client(self, **kwargs):
        return self.openai.OpenAI(**kwargs)

    def embed_query(self, text, **kwargs):
        kwargs.setdefault("extra_body", {"input_type": self.input_type})
        return super().embed_query(text, **kwargs)

llm = ChatNVIDIA(
    model="nvidia/nemotron-3-super-120b-a12b",
    base_url=NVIDIA_BASE_URL,
    api_key=os.getenv("NVIDIA_API_KEY"),
    model_kwargs={"response_format": {"type": "json_object"}},
)

embedder = NemotronEmbeddings(
    model="nvidia/nemotron-3-embed-1b",
    base_url=NVIDIA_BASE_URL,
    api_key=os.getenv("NVIDIA_API_KEY"),
    input_type="passage",
)

In [ ]:
from openai import OpenAI

nvidia_client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.environ.get("NVIDIA_API_KEY"),
)

In [ ]:
from langfuse.langchain import CallbackHandler
from langfuse import get_client

os.environ["LANGFUSE_PUBLIC_KEY"] = os.getenv("LANGFUSE_PUBLIC_KEY")
os.environ["LANGFUSE_SECRET_KEY"] = os.getenv("LANGFUSE_SECRET_KEY")
# Use the host that matches your keys' region. US keys (us.cloud.langfuse.com)
# will NOT report to the EU endpoint (cloud.langfuse.com) and vice-versa.
# Reads LANGFUSE_BASE_URL/LANGFUSE_HOST from .env, defaulting to the US region.
os.environ["LANGFUSE_HOST"] = (
    os.getenv("LANGFUSE_BASE_URL")
    or os.getenv("LANGFUSE_HOST")
    or "https://us.cloud.langfuse.com"
)

langfuse = get_client()

# Verify connection
if langfuse.auth_check():
    print("Langfuse client is authenticated and ready!")
    print(os.environ["LANGFUSE_HOST"])
else:
    print("Authentication failed. Please check your credentials and host.")

langfuse_handler = CallbackHandler()

Langfuse client is authenticated and ready!
https://cloud.langfuse.com


In [ ]:
import langfuse
print("version:", langfuse.__version__)
print("auth   :", langfuse.get_client().auth_check() if hasattr(langfuse, "get_client") else "n/a")

client = langfuse.get_client()
if hasattr(client, "start_as_current_span"):
    with client.start_as_current_span(name="manual-test-trace") as span:
        span.update(input={"hello": "world"}, output={"ok": True})
elif hasattr(client, "trace"):
    client.trace(name="manual-test-trace", input={"hello": "world"}, output={"ok": True})
client.flush()
print("✅ sent")

version: 4.14.1
auth   : True
✅ sent


In [ ]:

from langchain_classic.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
    HumanMessagePromptTemplate,
    AIMessagePromptTemplate,
    PromptTemplate,
)
import re
import asyncio
from typing import List, Union
from langchain_classic.agents import AgentOutputParser, AgentExecutor, create_react_agent
from langchain_classic.schema import AgentAction, AgentFinish



# 1‶  ━━━ Robust multi-action parser ━━━━━━━━━━━━
class MultiActionOutputParser(AgentOutputParser):
    """
    Extract *all* Action / Action Input pairs, even when inputs are
    multi-line or there is text between blocks.
    """
    _ACTION_RE = re.compile(
        r"""Action\s*\d*:\s*(.*?)\s*\n        # tool name
            Action\ Input\s*\d*:\s*([\s\S]*?) # tool input (non-greedy)
            (?=\nAction|\nObservation|\nFinal|\Z)  # stop at next block
        """,
        flags=re.IGNORECASE | re.VERBOSE,
    )

    def parse(self, text: str) -> Union[List[AgentAction], AgentFinish]:
        # ---------- collect all tool calls ----------
        matches = self._ACTION_RE.findall(text)
        if matches:
            return [
                AgentAction(
                    tool=tool.strip(),
                    tool_input=tool_input.strip().strip('"'),
                    log=text,
                )
                for tool, tool_input in matches
            ]

        # ---------- or a final answer ----------
        # final = re.search(r"(?:Final Answer|Final Thought):\s*(.*)", text, flags=re.IGNORECASE | re.DOTALL)
        final = re.search(r"(?:Final Answer):\s*(.*)", text, flags=re.IGNORECASE | re.DOTALL)

        # final = re.search(r"(?i)Final Answer:\s*(?:\*{2})?\s*\n+(.*)", text, flags=re.IGNORECASE | re.DOTALL)   ###PushToQA 04-07-2025 fixed ** issue

        if final:
            return AgentFinish(
                return_values={"output": final.group(1).strip()}, log=text
            )

        # ---------- fallback if no Action or Final Answer but appears like final output ----------
        if text.strip():  # fallback: any non-empty text is assumed to be the final answer
            return AgentFinish(
                return_values={"output": text.strip()}, log=text
            )

        raise ValueError(f"Could not parse agent output: {text!r}")


# 2‶  ━━━ Executor that really runs in parallel ━━━━━━━━━━━
class ParallelAgentExecutor(AgentExecutor):
    async def _aiter_next_step(self, name_to_tool_map, inputs):
        llm_output = await self.agent.aplan(
            intermediate_steps=self.intermediate_steps, **inputs
        )
        parsed = self.output_parser.parse(llm_output)

        # ----- run *all* Actions concurrently -----
        if isinstance(parsed, list):
            coros = [self._aperform_agent_action(name_to_tool_map, a) for a in parsed]
            results = await asyncio.gather(*coros)

            for a, r in zip(parsed, results):
                self.intermediate_steps.append((a, r))
                yield f"Observation: {r}\n"

        elif isinstance(parsed, AgentFinish):
            self.intermediate_steps.append(
                (parsed, parsed.return_values["output"])
            )
            yield f"Final Answer: {parsed.return_values['output']}\n"


# 3‶  ━━━ Helper to build the agent + prompt ━━━━━━━━━━━
def get_react_agent(llm, tools, system_prompt, verbose=False):
    """Helper function for creating agent executor"""
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        MessagesPlaceholder(variable_name="conversation_history", optional=True),
        HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], template='{input}')),
        AIMessagePromptTemplate(
            prompt=PromptTemplate(input_variables=['agent_scratchpad'], template='{agent_scratchpad}'))
    ])
    agent = create_react_agent(llm, tools, prompt, output_parser=MultiActionOutputParser(), stop_sequence=None)

    executor = ParallelAgentExecutor(agent=agent,
                                     tools=tools,
                                     verbose=verbose,
                                     stream_runnable=True,
                                     handle_parsing_errors=True,
                                     max_iterations=20,
                                     return_intermediate_steps=True,
                                     )
    return executor

In [ ]:
from tools_too import text2cypher_tool, hybrid_cypher_tool, global_search_tool, local_search_tool, web_search_tool

/usr/local/lib/python3.12/dist-packages/langchain_nvidia_ai_endpoints/_common.py:250: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


In [ ]:
from IPython.display import Markdown
import json
import prompts


def run_agentic_kag(query):
    """
    Runs the answer using react agent and provided tools.
    """
    # Run Hybrid Retrieval using all the available tools
    hybrid_agent = get_react_agent(
        llm,
        [text2cypher_tool, hybrid_cypher_tool, global_search_tool, local_search_tool, web_search_tool],
        prompts.REACT_PROMPT,
        verbose=True
    )
    hybrid_input = json.dumps({
        "query": query,
    })
    answer = hybrid_agent.invoke({"input": hybrid_input,
                                  "SYSTEM_PROMPT" : prompts.AV_SYSTEM_PROMPT,
                                  }, config={"callbacks": [langfuse_handler]}
                                 )
    final_answer = answer['output']
    # Push buffered traces to Langfuse so they show up in the app promptly.
    # langfuse.flush()
    return final_answer

In [ ]:
query = "Show me the top 10 highest-rated movies"
answer = run_agentic_kag(query)
display(Markdown(answer))



> Entering new ParallelAgentExecutor chain...
{
  "query": "Show me the top 10 highest-rated movies"
}

> Finished chain.


{
  "query": "Show me the top 10 highest-rated movies"
}

In [ ]:
query = "Suggest movies involving artificial intelligence, consciousness, or sentient machines."
answer = run_agentic_kag(query)
display(Markdown(answer))



> Entering new ParallelAgentExecutor chain...
{
  "query": "Suggest movies involving artificial intelligence, consciousness, or sentient machines."
}

> Finished chain.


{
  "query": "Suggest movies involving artificial intelligence, consciousness, or sentient machines."
}